In [1]:
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, csr_matrix, eye as speye
from scipy.sparse.linalg import eigsh
import os

print('imports ok')

imports ok


In [3]:
# load ds4x files
files = {
    'mosaic' : 'data/ds4x_mosaic.tif',
    'dem'    : 'data/ds4x_dem.tif',
    'ddem'   : 'data/ds4x_ddem.tif',
    'slope'  : 'data/ds4x_slope.tif',
    'aspect' : 'data/ds4x_aspect.tif',
} 

rasters = {}
metas   = {}

for name, path in files.items():
    with rasterio.open(path) as src:
        rasters[name] = src.read().astype(np.float32)
        metas[name]   = {
            'shape'    : (src.height, src.width),
            'transform': src.transform,
            'crs'      : src.crs,
            'nodata'   : src.nodata,
            'count'    : src.count
        }

    print(f'{name:>8}: shape={rasters[name].shape}  '
          f'nodata={metas[name]["nodata"]}')
        

  mosaic: shape=(7, 470, 491)  nodata=-32768.0
     dem: shape=(1, 470, 491)  nodata=-9999.0
    ddem: shape=(1, 470, 491)  nodata=-9999.0
   slope: shape=(1, 470, 491)  nodata=-9999.0
  aspect: shape=(1, 470, 491)  nodata=-9999.0


In [5]:
# confirm rasters share the same grid
shapes = [metas[k]['shape'] for k in files]
assert len(set(shapes)) == 1, f'shape mismatch: {shapes}'
print()
print(f' rasters confirmed: {shapes[0][0]} rows x {shapes[0][1]} cols')


 rasters confirmed: 470 rows x 491 cols


In [6]:
# set metadata for processing
NROWS, NCOLS = shapes[0]
transform    = metas['mosaic']['transform']
crs          = metas['mosaic']['crs']
RES_M        = 4.068 ## HARDCODE

print(f'resolution = {RES_M} m')
print(f'crs: {crs.to_epsg()}')

resolution = 4.068 m
crs: 32606


In [11]:
# uniform nodata and build clean arrays

SCALE = 32768.0

# ---- mosaic: 7 bands, nodata = -32768
mosaic = rasters['mosaic'].copy()
mosaic[mosaic == -32768.0] = np.nan

# scale reflectance bands 1-6 (np 0-5), leave lwir in DN
for b in range(6):
    mosaic[b] = mosaic[b] / SCALE

# ---- DEM ----
dem = rasters['dem'][0].copy()
dem[dem == -9999.0] = np.nan

# --- dDEM ----
# flip sign so negative is ice loss
ddem = rasters['ddem'][0].copy()
ddem[ddem == -9999.0] = np.nan
ddem = -ddem

# --- slope ---
slope = rasters['slope'][0].copy()
slope[slope == -9999.0] = np.nan

# -- aspect: convert to sin/cos --
aspect_raw = rasters['aspect'][0].copy()
aspect_raw[aspect_raw == -9999.0] = np.nan
aspect_sin = np.sin(np.deg2rad(aspect_raw))
aspect_cos = np.cos(np.deg2rad(aspect_raw))

print('Array ranges after nodata handling:')
print(f'  mosaic band 0 (Blue): [{np.nanmin(mosaic[0]):.4f}, {np.nanmax(mosaic[0]):.4f}]')
print(f'  mosaic band 6 (LWIR): [{np.nanmin(mosaic[6]):.1f}, {np.nanmax(mosaic[6]):.1f}]')
print(f'  dem   : [{np.nanmin(dem):.2f}, {np.nanmax(dem):.2f}] m')
print(f'  ddem  : [{np.nanmin(ddem):.2f}, {np.nanmax(ddem):.2f}] m')
print(f'  slope : [{np.nanmin(slope):.2f}, {np.nanmax(slope):.2f}] deg')
print(f'  asin  : [{np.nanmin(aspect_sin):.3f}, {np.nanmax(aspect_sin):.3f}]')
print(f'  acos  : [{np.nanmin(aspect_cos):.3f}, {np.nanmax(aspect_cos):.3f}]')
print()

# NaN counts
for name, arr in [('mosaic b0', mosaic[0]), ('dem', dem),
                   ('ddem', ddem), ('slope', slope), ('aspect_sin',aspect_sin)]:
    print(f'  {name:>10} NaNs: {np.isnan(arr).sum():,}')

Array ranges after nodata handling:
  mosaic band 0 (Blue): [0.0169, 0.7316]
  mosaic band 6 (LWIR): [27558.0, 29915.0]
  dem   : [813.41, 986.90] m
  ddem  : [-88.07, 31.82] m
  slope : [0.00, 67.86] deg
  asin  : [-1.000, 1.000]
  acos  : [-1.000, 1.000]

   mosaic b0 NaNs: 105,248
         dem NaNs: 102,671
        ddem NaNs: 1,150
       slope NaNs: 104,968
  aspect_sin NaNs: 104,968


In [13]:
# set valid pixels
# pixel valid if:
#    1. all 5 reflectance bands (0-4) non-nan
#    2. dem is non-nan
#    3. slope/aspect are non-nan
# dDEM nans handled separately
#    -- don't exclude pixels for missing ddem


spectral_valid = ~np.isnan(mosaic).any(axis=0) # (470, 491)
topo_valid     = ~np.isnan(dem) & ~np.isnan(slope) # (470, 491)
valid_mask_2d  = spectral_valid & topo_valid

print(f'Spectral valid : {spectral_valid.sum():,}')
print(f'Topo valid     : {topo_valid.sum():,}')
print(f'Combined valid : {valid_mask_2d.sum():,}')
print(f'Coverage       : {100*valid_mask_2d.mean():.1f}%')
print()

Spectral valid : 125,522
Topo valid     : 125,802
Combined valid : 124,720
Coverage       : 54.0%



In [14]:
# build index map: -1 = invalid, >= 0 = node index
index_map = np.full((NROWS, NCOLS), -1, dtype=np.float32)
valid_positions = np.where(valid_mask_2d)
n_valid = len(valid_positions[0])
index_map[valid_positions] = np.arange(n_valid)

In [19]:
# node coordinates
valid_r = valid_positions[0]
valid_c = valid_positions[1]

In [21]:
print(f'N valid nodes  : {n_valid:,}')
print(f'Index map range: [{index_map.min()}, {index_map.max()}]')
print()

N valid nodes  : 124,720
Index map range: [-1.0, 124719.0]



In [22]:
# save
np.save('data/ds4x_index_map.npy', index_map)
np.save('data/ds4x_valid_r.npy',   valid_r)
np.save('data/ds4x_valid_c.npy',   valid_c)
print('saved')

saved


In [26]:
### extract node feature arrays ###
# node order is row-major as assigned by index map #

# spectral: 7 bands, reflectance scaled
spec7 = np.stack([
    mosaic[b, valid_r, valid_c] for b in range(7)
], axis=1).astype(np.float32)   # (N, 7)

# topographic features
elev_nodes  = dem       [valid_r, valid_c].astype(np.float32)
slope_nodes = slope     [valid_r, valid_c].astype(np.float32)
asin_nodes  = aspect_sin[valid_r, valid_c].astype(np.float32)
acos_nodes  = aspect_cos[valid_r, valid_c].astype(np.float32)
ddem_nodes  = ddem      [valid_r, valid_c].astype(np.float32)

# stack topographic features
# elevation, slope, aspect_sin, aspect_cos, ddem
topo5 = np.stack([
    elev_nodes, slope_nodes, asin_nodes, acos_nodes, ddem_nodes
], axis=1)     # (N, 5)

# check
N = spec7.shape[0]
print('check:')
print(f'N (len spec7): {N:,}')
print(f'len topo5: {topo5.shape[0]:,}')

check:
N (len spec7): 124,720
len topo5: 124,720


In [27]:

band_names_7 = ['blue', 'green', 'red', 'rededge', 'nir', 'panc', 'lwir']
topo_names   = ['elevation', 'slope', 'aspect_sin', 'aspect_cos', 'ddem']

print(f'{"Feature":>14}  {"Min":>10}  {"Max":>10}  {"Mean":>10}  {"NaNs":>8}')
print('-' * 58)
for i, name in enumerate(band_names_7):
    col = spec7[:, i]
    print(f'{name:>14}  {np.nanmin(col):>10.4f}  {np.nanmax(col):>10.4f}  '
          f'{np.nanmean(col):>10.4f}  {np.isnan(col).sum():>8,}')

print()
for i, name in enumerate(topo_names):
    col = topo5[:, i]
    print(f'{name:>14}  {np.nanmin(col):>10.4f}  {np.nanmax(col):>10.4f}  '
          f'{np.nanmean(col):>10.4f}  {np.isnan(col).sum():>8,}')

       Feature         Min         Max        Mean      NaNs
----------------------------------------------------------
          blue      0.0169      0.7316      0.2309         0
         green      0.0243      0.7462      0.2292         0
           red      0.0139      0.7853      0.2330         0
       rededge      0.0573      0.9270      0.3201         0
           nir      0.0302      0.7473      0.3000         0
          panc      0.0000      1.0000      0.3773         0
          lwir  27558.0000  29915.0000  28992.9258         0

     elevation    813.4139    986.9013    861.2152         0
         slope      0.0035     67.8577     13.2020         0
    aspect_sin     -1.0000      1.0000     -0.0227         0
    aspect_cos     -1.0000      1.0000      0.2569         0
          ddem    -88.0747     17.6390    -16.8060       298


In [28]:
# save 
np.save('data/ds4x_spec7.npy', spec7)
np.save('data/ds4x_topo5.npy', topo5)
print('saved')

saved


In [29]:
### build laplacians ###
### and edge list    ###

In [30]:
# connectivity edge list #
offsets = [(-1,-1),(-1,0),(-1,1),
           ( 0,-1),        (0,1),
           ( 1,-1),( 1,0),( 1,1)]

rows_i, rows_j = [], []

for di, dj in offsets:
    r0, r1 = max(0, -di), min(NROWS, NROWS - di)
    c0, c1 = max(0, -dj), min(NCOLS, NCOLS - dj)

    src = index_map[r0:r1, c0:c1]
    nbr = index_map[r0+di:r1+di, c0+dj:c1+dj]

    valid = (src >= 0) & (nbr >= 0)
    rows_i.append(src[valid])
    rows_j.append(nbr[valid])

edge_i = np.concatenate(rows_i).astype(np.int32)
edge_j = np.concatenate(rows_j).astype(np.int32)

print(f'Total directed edges : {len(edge_i):,}')
print(f'Total undirected edges: {len(edge_i)//2:,}')
print(f'Avg neighbors per node: {len(edge_i)/N:.2f}')
print(f'Min node index: {min(edge_i.min(), edge_j.min())}')
print(f'Max node index: {max(edge_i.max(), edge_j.max())}')
print(f'Expected max  : {N-1}')

Total directed edges : 992,376
Total undirected edges: 496,188
Avg neighbors per node: 7.96
Min node index: 0
Max node index: 124719
Expected max  : 124719


In [31]:
# save
np.save('data/ds4x_edge_i.npy', edge_i)
np.save('data/ds4x_edge_j.npy', edge_j)
print('saved')

saved


In [32]:
### empirical beta calibration for laplacians ###

# extract signal values at edge endpoints
elev_i = topo5[edge_i, 0]
elev_j = topo5[edge_j, 0]

asin_i = topo5[edge_i, 2]
asin_j = topo5[edge_j, 2]
acos_i = topo5[edge_i, 3]
acos_j = topo5[edge_j, 3]

ddem_i = topo5[edge_i, 4]
ddem_j = topo5[edge_j, 4]

# pairwise differences
elev_diff2   = (elev_i - elev_j)**2
aspect_dist2 = (asin_i - asin_j)**2 + (acos_i - acos_j)**2
ddem_diff2   = (ddem_i - ddem_j)**2

# nan mask for ddam
ddem_nan = np.isnan(ddem_i) | np.isnan(ddem_j)
valid_ddem = ~ddem_nan

print('Elevation neighbour differences (metres):')
elev_diffs = np.sqrt(elev_diff2)
for p in [50, 75, 90, 95, 99]:
    print(f'  p{p:>2}: {np.percentile(elev_diffs, p):.4f} m')

print()
print('dDEM neighbour differences (metres):')
ddem_diffs = np.sqrt(ddem_diff2[valid_ddem])
for p in [50, 75, 90, 95, 99]:
    print(f'  p{p:>2}: {np.percentile(ddem_diffs, p):.4f} m')

Elevation neighbour differences (metres):
  p50: 0.5652 m
  p75: 1.1605 m
  p90: 1.8857 m
  p95: 2.3671 m
  p99: 3.3579 m

dDEM neighbour differences (metres):
  p50: 0.6622 m
  p75: 1.3223 m
  p90: 2.1932 m
  p95: 2.8665 m
  p99: 4.5755 m

Fraction of edges with weight < 0.5 at candidate beta values:
      beta    elev (%)    aspect (%)    ddem (%)
------------------------------------------------


In [34]:
# gaussian kernel w = exp(-β · d²)
# converts a signal difference d between neighbors
# into a weight w
# weight near 1 -> strong connection - enforce continuity
#        model penalized if membership values very different
# weight near 0 -> weak connection - allows discontinuity
#        model free to have different memberships
# beta controls sensitivity - how large
#   must difference be before edge gets suppressed

# edge weight < 0.5 -> model more free than constrained
# 0.5 natural threshold for "edge substantially suppressed

In [33]:
def frac_below_half(diff2, beta):
    w = np.exp(-beta * diff2)
    return 100 * (w < 0.5).mean()

print()
print('Fraction of edges with weight < 0.5 at candidate beta values:')
print(f'{"beta":>10}  {"elev (%)":>10}  {"aspect (%)":>12}  {"ddem (%)":>10}')
print('-' * 48)

for beta in [0.005, 0.05, 0.5, 2.0, 5.0, 10.0, 20.0]:
    fe = frac_below_half(elev_diff2, beta)
    fa = frac_below_half(aspect_dist2, beta)
    fd = frac_below_half(ddem_diff2[valid_ddem], beta)
    print(f'{beta:>10.3f}  {fe:>10.2f}  {fa:>12.2f}  {fd:>10.2f}')


Fraction of edges with weight < 0.5 at candidate beta values:
      beta    elev (%)    aspect (%)    ddem (%)
------------------------------------------------
     0.005        0.00          0.00        0.00
     0.050        0.54          0.00        2.17
     0.500       24.50          8.32       29.08
     2.000       48.65         20.40       54.14
     5.000       62.62         31.94       68.18
    10.000       71.19         41.91       76.61
    20.000       77.83         52.12       83.03


In [38]:
## fraction of suppressed edges indicates
# how much of the graph actively permits discontinuities
# too few supppressed edges (beta too small):
#   laplacian nearly uniform - smooths everywhere equally
# too many suppressed edges (beta too large):
#   graph nearly disconnected -- almost no smoothness anywhere

# note: "suppressed" : "smoothness penalty is suppressed"
#   i.e. weight is small -> pixels allowed to differ

# target 20-35% suppressed edges  -- heuristic
#    "graph is doing meaningful work"

In [36]:
### starting beta values ###
BETA_ELEV   = 0.5
BETA_ASPECT = 2.0
BETA_DDEM   = 0.5

In [37]:
### compute edge weights ###

# elevation weights
w_elev = np.exp(-BETA_ELEV * elev_diff2).astype(np.float32)

# aspect weights
w_aspect = np.exp(-BETA_ASPECT * aspect_dist2).astype(np.float32)

# ddem weights -- zero out NAN edges
w_ddem = np.exp(-BETA_DDEM * ddem_diff2).astype(np.float32)
w_ddem[ddem_nan] = 0.0

print('edge weight statistics:')
print(f'{"":12}  {"min":>8}  {"max":>8}  {"mean":>8}  {"<0.5 edges":>12}')
print('-' * 56)
for name, w in [('w_elev', w_elev),
                ('w_aspect', w_aspect),
                ('w_ddem', w_ddem)]:
    below = (w < 0.5).sum()
    print(f'{name:12}  {w.min():>8.4f}  {w.max():>8.4f}  '
          f'{w.mean():>8.4f}  {below:>12,}')

edge weight statistics:
                   min       max      mean    <0.5 edges
--------------------------------------------------------
w_elev          0.0000    1.0000    0.7148       243,148
w_aspect        0.0003    1.0000    0.7538       202,460
w_ddem          0.0000    1.0000    0.6677       290,834


In [41]:
#### BUILD NORMALIZED LAPLACIANS ###

def build_normalized_laplacian(edge_i, edge_j, weights, n):
    A = coo_matrix(
        (weights, (edge_i, edge_j)),
        shape=(n,n)
    ).tocsr()

    d = np.array(A.sum(axis=1)).ravel()
    d_safe      = np.where(d > 0, d, 1.0)
    d_invsqrt   = np.where(d > 0, 1.0/np.sqrt(d_safe), 0.0)

    D_invsqrt = csr_matrix(
        (d_invsqrt, (np.arange(n), np.arange(n))),
        shape=(n,n)
    )

    L = speye(n, format='csr') - D_invsqrt @ A @ D_invsqrt

    n_isolated = (d == 0).sum()
    if n_isolated > 0:
        print(f'isolated nodes: {n_isolated}')

    return L, d

In [42]:
print('building L_elev...')
L_elev, d_elev = build_normalized_laplacian(edge_i, edge_j, w_elev, N)
print(f'  nnz: {L_elev.nnz:,}')

building L_elev...
  nnz: 1,117,088


In [43]:
print('building L_aspect...')
L_aspect, d_aspect = build_normalized_laplacian(edge_i, edge_j, w_aspect, N)
print(f' nnz: {L_aspect.nnz:,}')

building L_aspect...
 nnz: 1,117,096


In [44]:
print('building L_ddem...')
L_ddem, d_ddem = build_normalized_laplacian(edge_i, edge_j, w_ddem, N)
print(f' nnz: {L_ddem.nnz:,}')

building L_ddem...
isolated nodes: 298
 nnz: 1,113,956


In [45]:

print()
print('Degree statistics:')
print(f'  {"":10}  {"mean":>8}  {"min":>10}  {"max":>8}')
print('-' * 42)
for name, d in [('elev', d_elev),
                ('aspect', d_aspect),
                ('ddem', d_ddem)]:
    print(f'  {name:10}  {d.mean():>8.3f}  {d.min():>10.6f}  {d.max():>8.3f}')


Degree statistics:
                  mean         min       max
------------------------------------------
  elev           5.687    0.039837     8.000
  aspect         5.998    0.008177     8.000
  ddem           5.312    0.000000     7.994


In [46]:
# save
import scipy.sparse as sp
sp.save_npz('data/ds4x_L_elev.npz',   L_elev)
sp.save_npz('data/ds4x_L_aspect.npz', L_aspect)
sp.save_npz('data/ds4x_L_ddem.npz',   L_ddem)
print('saved')

saved


In [47]:
# normalize features and build X matrix
# pixel is spectrally valid if all 7 bands non-nan
# already by index_map construction but confirm

spectral_valid_mask = ~np.isnan(spec7).any(axis=1)
print(f'spectrally valid: {spectral_valid_mask.sum():,} / {N:,}')
print()

spectrally valid: 124,720 / 124,720



In [48]:
# z-score normalization using valid pixels only
def zscore(data, mask):
    valid = data[mask]
    mu = valid.mean(axis=0)
    std = valid.std(axis=0)
    std = np.where(std > 1e-8, std, 1.0)
    return ((data - mu) / std).astype(np.float32), mu, std

In [49]:
# normalize spectral (7 bands)
spec7_norm, spec_mu, spec_std = zscore(spec7, spectral_valid_mask)
print('spectrals normalized')

spectrals normalized


In [51]:
# normalize topo (5 features: elev, slope, asin, acos, ddem)
# fill ddem nans with 0 before normalization
topo5_filled = topo5.copy()
ddem_nan_nodes = np.isnan(topo5_filled[:,4])
topo5_filled[ddem_nan_nodes, 4] = 0.0
topo5_norm, topo_mu, topo_std = zscore(topo5_filled, spectral_valid_mask)
print('topos normalized')

topos normalized


In [52]:
# concatenate: X = (N, 12) = 7 spec + 5 topo

X_norm = np.concatenate([spec7_norm, topo5_norm], axis=1)

# zero out spectrally non valid pixels
nan_rows = ~spectral_valid_mask
X_norm[nan_rows] = 0.0

print(f'X_norm shape: {X_norm.shape}')
print()

# verify
valid_X = X_norm[spectral_valid_mask]
print(f'Column means (should be ~0): '
      f'min={valid_X.mean(axis=0).min():.4f}  '
      f'max={valid_X.mean(axis=0).max():.4f}')
print(f'Column stds  (should be ~1): '
      f'min={valid_X.std(axis=0).min():.4f}  '
      f'max={valid_X.std(axis=0).max():.4f}')
print()

X_norm shape: (124720, 12)

Column means (should be ~0): min=-0.0010  max=0.0000
Column stds  (should be ~1): min=1.0000  max=1.0000



In [53]:
norm_params = {
    'spec_mu': spec_mu, 'spec_std': spec_std,
    'topo_mu': topo_mu, 'topo_std': topo_std,
    'feature_order': ['Blue','Green','Red','RedEdge','NIR','Panc','LWIR',
                      'elevation','slope','aspect_sin','aspect_cos','ddem']
}

In [54]:
# save
np.save('data/ds4x_X_norm.npy',  X_norm)
np.save('data/ds4x_spec7.npy',   spec7)         # raw reflectance for loss
np.save('data/ds4x_specvalid_mask.npy', spectral_valid_mask)

# save params
np.save('data/ds4x_norm_params.npy', norm_params, allow_pickle=True)
print('saved')

saved


In [58]:
# build L_band from endmember spectra

# load anchor (endmember) matrix
E = np.load('data/anc_endmembers5x7.npy')  # (5 anchors, 7 bands)
E_df = pd.read_csv('data/anc_endmems5x7_named.csv')

print('anchor endmember matrix E:')
print(E_df[['anchor_id','name']].to_string())
print()

# use bands 1-6 (blue-panc, np 0-5) for band correlation
# exclude lwir (band 7) -- different physical scale
E_6band = E[:, :6].astype(np.float64) # (5 anchors, 6 bands)

# compute correlation matrix across anchors
# here correlate bands (columns) so transpose first
E_centered = E_6band - E_6band.mean(axis=0)
std = E_6band.std(axis=0)
# 6x6 band correlations / total - lwir
C = (E_centered.T @ E_centered) / (E_6band.shape[0]- 1) 
C = C / (std[:, None] * std[None, :])

band_names_6 = ['blue', 'green', 'red', 'rededge', 'nir', 'panc']
print('band correlation matrix (from anchor spectra):')
print(f'{"":>8}', end='')
for name in band_names_6:
    print(f'{name:>10}', end='')
print()
for i, name in enumerate(band_names_6):
    print(f'{name:>8}', end='')
    for j in range(6):
        print(f'{C[i,j]:>10.4f}', end='')
    print()
print()

anchor endmember matrix E:
   anchor_id                                      name
0         18             turbid glacier outflow (gray)
1         26                     gray outwash sediment
2         13                    north slope vegetation
3          2                        south slope gravel
4          5  post-glacial settling (north slope edge)

band correlation matrix (from anchor spectra):
              blue     green       red   rededge       nir      panc
    blue    1.2500    1.2324    1.1901    0.8813    0.1245   -0.4801
   green    1.2324    1.2500    1.1722    0.9748    0.3206   -0.2808
     red    1.1901    1.1722    1.2500    0.9468    0.2293   -0.4393
 rededge    0.8813    0.9748    0.9468    1.2500    0.8077    0.2711
     nir    0.1245    0.3206    0.2293    0.8077    1.2500    1.0662
    panc   -0.4801   -0.2808   -0.4393    0.2711    1.0662    1.2500



In [62]:
# edge weights: positive correlations only
W_band = np.maximum(C, 0).astype(np.float32)
np.fill_diagonal(W_band, 0)
print('band edge weight matrix calced')

band edge weight matrix calced


In [63]:
# normalized laplacian (no sparsity needed)
d = W_band.sum(axis=1)
d_safe    = np.where(d>0, d, 1.0)
d_invsqrt = 1.0 / np.sqrt(d_safe)
D_invsqrt = np.diag(d_invsqrt)
L_band = np.eye(6) - D_invsqrt @ W_band @ D_invsqrt
print('L_band calced')

L_band calced


In [64]:
# verify
eigenvalues = np.linalg.eigvalsh(L_band)
print(f'L_band shape     : {L_band.shape}')
print(f'Eigenvalue range : [{eigenvalues.min():.6f}, {eigenvalues.max():.6f}]')
print(f'Symmetry check   : {np.abs(L_band - L_band.T).max():.2e}')

L_band shape     : (6, 6)
Eigenvalue range : [-0.000000, 1.599187]
Symmetry check   : 2.98e-08


In [65]:
## 5-anchor correlation matrix has values>1.0 on diagonal
# and some off-diagonal > 1.0 -- not valid correlation matrix
# issue: 5 anchors and 6 bands - rank-deficient 
# need at least as many observations as variables
# ideally many more
# fix: compute band correlation from full mosaic data:

In [66]:
# compute band correlation from full mosaic:
S = spec7[spectral_valid_mask, :6].astype(np.float64) # (N_valid, 6)

S_centered = S - S.mean(axis=0)
std = S.std(axis=0)
C = (S_centered.T @ S_centered) / (S.shape[0] - 1)
C = C / (std[:, None] * std[None, :])

band_names_6 = ['blue', 'green', 'red', 'rededge', 'nir', 'panc']
print('band correlation matrix (from anchor spectra):')
print(f'{"":>8}', end='')
for name in band_names_6:
    print(f'{name:>10}', end='')
print()
for i, name in enumerate(band_names_6):
    print(f'{name:>8}', end='')
    for j in range(6):
        print(f'{C[i,j]:>10.4f}', end='')
    print()
print()

band correlation matrix (from anchor spectra):
              blue     green       red   rededge       nir      panc
    blue    1.0000    0.9723    0.9890    0.8178    0.4169   -0.2755
   green    0.9723    1.0000    0.9779    0.8844    0.5896   -0.0757
     red    0.9890    0.9779    1.0000    0.8413    0.4819   -0.2161
 rededge    0.8178    0.8844    0.8413    1.0000    0.6946    0.1518
     nir    0.4169    0.5896    0.4819    0.6946    1.0000    0.7357
    panc   -0.2755   -0.0757   -0.2161    0.1518    0.7357    1.0000



In [67]:
# (recompute) normalized laplacian (no sparsity needed)
d = W_band.sum(axis=1)
d_safe    = np.where(d>0, d, 1.0)
d_invsqrt = 1.0 / np.sqrt(d_safe)
D_invsqrt = np.diag(d_invsqrt)
L_band = np.eye(6) - D_invsqrt @ W_band @ D_invsqrt
print('L_band calced')

L_band calced


In [68]:
# verify
eigenvalues = np.linalg.eigvalsh(L_band)
print(f'L_band shape     : {L_band.shape}')
print(f'Eigenvalue range : [{eigenvalues.min():.6f}, {eigenvalues.max():.6f}]')
print(f'Symmetry check   : {np.abs(L_band - L_band.T).max():.2e}')

L_band shape     : (6, 6)
Eigenvalue range : [-0.000000, 1.599187]
Symmetry check   : 2.98e-08


In [69]:
# save
np.save('data/ds4x_L_band.npy', L_band)
print('saved')

saved
